In [21]:
import base64
import html
import ipywidgets as widgets
from IPython.display import HTML as IPyHTML, clear_output, display

# ---------------------------------------------------------------------
# 1. CANDIDATES & SETTINGS
# ---------------------------------------------------------------------
CANDIDATES = [
    {"name": "Rigathi Gachagua",        "initials": "RG", "party": "DCP", "color": "#6D28D9"},
    {"name": "Dr. William Ruto",        "initials": "WR", "party": "UDA", "color": "#0EA5E9"},
    {"name": "Edwin Sifuna",            "initials": "ES", "party": "Linda Mwananchi", "color": "#EC4899"},
    {"name": "Dr. Fred Matiang'i",      "initials": "FM", "party": "Jubilee", "color": "#F59E0B"},
    {"name": "Stephen Kalonzo Musyoka", "initials": "KM", "party": "Wiper", "color": "#10B981"},
]

# Computerized / digital network background image
BACKGROUND_URL = "https://images.unsplash.com/photo-1451187580459-43490279c0fa?q=80&w=1920&auto=format&fit=crop"

# ---------------------------------------------------------------------
# 2. HTML / CSS GENERATORS
# ---------------------------------------------------------------------
def card_result_html(c, votes, total):
    pct = (votes / total * 100) if total else 0.0
    return f"""
    <div style="text-align:right; font-family:'Poppins', sans-serif; min-width: 90px;">
      <div style="font-size:22px; font-weight:800; color:{c['color']}; line-height:1;">{pct:.1f}%</div>
      <div style="font-size:11px; color:#6b7280; margin:4px 0 6px;">{votes} vote{'s' if votes != 1 else ''}</div>
      <div style="height:6px; background:#eef0f4; border-radius:99px; overflow:hidden;">
        <div style="width:{pct:.1f}%; height:100%; border-radius:99px;
                    background:linear-gradient(90deg, {c['color']}, {c['color']}88);
                    transition:width .6s ease;"></div>
      </div>
    </div>"""

def results_panel_html(votes):
    total = sum(votes.values())
    order = sorted(range(len(CANDIDATES)), key=lambda i: -votes[i])
    rows = ""
    for i in order:
        c = CANDIDATES[i]
        pct = (votes[i] / total * 100) if total else 0.0
        rows += f"""
        <div style="margin-bottom:14px; font-family:'Poppins', sans-serif;">
          <div style="display:flex; justify-content:space-between; align-items:baseline;">
            <span style="font-weight:600; color:#1f2937; font-size:14px;">{html.escape(c['name'])}
              <span style="font-weight:400; color:#6b7280; font-size:12px;"> · {html.escape(c['party'])}</span></span>
            <span style="font-weight:700; color:{c['color']}; font-size:14px;">{pct:.1f}%
              <span style="font-weight:400; color:#6b7280; font-size:11px;"> ({votes[i]})</span></span>
          </div>
          <div style="height:8px; background:#eef0f4; border-radius:99px; overflow:hidden; margin-top:6px;">
            <div style="width:{pct:.1f}%; height:100%; border-radius:99px;
                        background:linear-gradient(90deg, {c['color']}, {c['color']}77);"></div>
          </div>
        </div>"""
    return f"""
    <div style="background:#fff; border-radius:16px; padding:20px 22px; margin-top:20px;
                box-shadow:0 8px 24px rgba(0,0,0,.08); font-family:'Poppins', sans-serif;">
      <div style="font-size:16px; font-weight:700; color:#1f2937; margin-bottom:14px;">📊 Live Results</div>
      {rows}
      <div style="font-size:12px; color:#6b7280; border-top:1px solid #eef0f4; padding-top:12px; text-align:right;">
        Total votes cast: <b>{total}</b></div>
    </div>"""

# ---------------------------------------------------------------------
# 3. GLOBAL CSS & LAYOUT
# ---------------------------------------------------------------------
BASE_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700;800&display=swap');

/* Computerized Blurred Background Layer */
body::before {{
    content: "";
    position: fixed;
    top: 0; left: 0; width: 100vw; height: 100vh;
    background-image: url('{BACKGROUND_URL}');
    background-size: cover;
    background-position: center;
    filter: blur(10px) brightness(0.6); /* Increased blur for the computerized look */
    z-index: -1;
}}

/* Centering the App */
.app-wrapper {{
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 100vh;
    padding: 20px;
    box-sizing: border-box;
    font-family: 'Poppins', sans-serif;
}}

/* Card Styling */
.poll-card {{
    background: #fff;
    border-radius: 16px;
    padding: 12px 18px;
    margin-bottom: 12px;
    box-shadow: 0 4px 14px rgba(0,0,0,.08);
    transition: transform .15s ease, box-shadow .15s ease, opacity .3s ease;
    cursor: pointer;
    border: 2px solid transparent;
}}
.poll-card.poll-open:hover {{
    transform: translateY(-2px);
    box-shadow: 0 8px 24px rgba(0,0,0,.15);
}}
.poll-card.poll-dim {{ opacity: .6; cursor: not-allowed; }}
.poll-card.poll-selected {{ border-color: #7C3AED; }}

/* Button Resets (hiding standard ipywidgets styling) */
.poll-card button:disabled, .poll-card .jupyter-button:disabled {{ opacity: 1 !important; cursor: default !important; }}
.poll-av.jupyter-button, .poll-av {{ border-radius: 50% !important; color: #fff !important; font-weight: 700 !important; font-size: 16px !important; border: none !important; box-shadow: none !important; }}
.poll-name.jupyter-button, .poll-name {{ background: transparent !important; border: none !important; box-shadow: none !important; color: #1f2937 !important; font-size: 16px !important; font-weight: 600 !important; text-align: left !important; justify-content: flex-start !important; padding: 0 !important; font-family: 'Poppins', sans-serif !important; }}
.poll-open .poll-name:hover {{ text-decoration: underline; }}
</style>
"""

HEADER_HTML = """
<div style="background:linear-gradient(135deg, #7C3AED 0%, #06B6D4 100%); color:#fff;
            border-radius:20px; padding:30px; margin-bottom:22px;
            box-shadow:0 12px 30px rgba(124,58,237,.4); font-family:'Poppins', sans-serif; text-align:center;">
  <div style="font-size:28px; font-weight:800; line-height:1.2;">🇰🇪 Public Opinion On Who Will Lead Kenya 2027</div>
  <div style="font-size:14px; opacity:.9; margin-top:8px; font-weight:400;">Tap a candidate card to cast your vote instantly.</div>
</div>
"""

# ---------------------------------------------------------------------
# 4. INTERACTIVE LOGIC
# ---------------------------------------------------------------------
votes = {i: 0 for i in range(len(CANDIDATES))}
state = {"voted": False}
result_boxes, cards, click_targets = [], [], []

status = widgets.Button(
    description="Tap a candidate card to vote", disabled=True,
    layout=widgets.Layout(width="100%", height="46px", margin="10px 0 0 0"))
status.style.font_family = "Poppins"

out = widgets.Output()
with out:
    display(IPyHTML('<div style="color:#e5e7eb; font-family:Poppins; text-align:center; margin-top:10px;">Results will appear here after you vote.</div>'))

def refresh():
    total = sum(votes.values())
    for i, box in enumerate(result_boxes):
        box.value = card_result_html(CANDIDATES[i], votes[i], total)
    with out:
        clear_output(wait=True)
        display(IPyHTML(results_panel_html(votes)))

def cast_vote(i):
    if state["voted"]: return
    state["voted"] = True
    votes[i] += 1

    # Lock all controls
    for b in click_targets: b.disabled = True
    for j, card in enumerate(cards):
        card.remove_class("poll-open")
        card.add_class("poll-selected" if j == i else "poll-dim")

    status.description = "✅ Vote Submitted"
    status.button_style = "success"
    status.disabled = True
    refresh()

def make_card(i, c):
    # Icon-style Avatar
    avatar = widgets.Button(
        description=c["initials"], tooltip=f"Vote for {c['name']}",
        layout=widgets.Layout(width="50px", height="50px", min_width="50px", margin="0 14px 0 0", padding="0"))
    avatar.style.button_color = c["color"]
    avatar.add_class("poll-av")

    # Candidate Name
    name_btn = widgets.Button(
        description=c["name"], tooltip=f"Vote for {c['name']}",
        layout=widgets.Layout(width="100%", height="30px"))
    name_btn.add_class("poll-name")

    # Party Chip
    party_chip = widgets.HTML(
        f'<div style="display:flex; align-items:center; gap:6px; color:#6b7280; font-size:12px; font-family:Poppins;">'
        f'<span style="width:12px; height:12px; border-radius:3px; background:{c["color"]}; display:inline-block;"></span>'
        f'{html.escape(c["party"])}</div>'
    )

    for b in (avatar, name_btn):
        b.on_click(lambda _b, i=i: cast_vote(i))
        click_targets.append(b)

    result = widgets.HTML("", layout=widgets.Layout(width="120px", min_width="120px"))
    result_boxes.append(result)

    middle = widgets.VBox([name_btn, party_chip], layout=widgets.Layout(flex="1 1 auto", min_width="0"))
    card = widgets.HBox([avatar, middle, result], layout=widgets.Layout(width="100%", align_items="center"))

    card.add_class("poll-card")
    card.add_class("poll-open")
    cards.append(card)
    return card

card_widgets = [make_card(i, c) for i, c in enumerate(CANDIDATES)]

# ---------------------------------------------------------------------
# 5. ASSEMBLE AND CENTER THE APP
# ---------------------------------------------------------------------
# The inner container with max-width
app_content = widgets.VBox(
    [widgets.HTML(HEADER_HTML), *card_widgets, status, out],
    layout=widgets.Layout(max_width="600px", width="100%")
)

# Wrap everything in the CSS flexbox to center it on the page
ui = widgets.VBox(
    [widgets.HTML(BASE_CSS), app_content],
    layout=widgets.Layout(width="100%", align_items="center", justify_content="center")
)
ui.add_class("app-wrapper")

display(ui)